# Validate Structured GLOT vs Original Code

This notebook validates that our **structured GLOT implementation** (after adopting the original architecture) produces results consistent with the **original monolithic code** from [ipsitmantri/GLOT](https://github.com/ipsitmantri/GLOT).

### What changed in the adoption
| Component | Original Code | Our Structured Code |
|-----------|--------------|-------------------|
| Graph construction | Cosine sim + threshold, edge weights | `graph_construction.py` — same |
| GNN | No input proj, GATConv, JK cat = d + K×p | `token_gnn.py` — same, + GCN/GIN/GINE |
| Readout | Scorer hidden = configurable | `readout.py` — max(128, dim//2) |
| Pooler | Single file `main.py` | `model.py` (GLOTPooler) |

### Experiment design
- **Backbones:** `bert-base-uncased`, `roberta-base` (matching original)
- **Hyperparameters:** Match original code exactly (hidden=256, layers=4, τ=0.3, epochs=3, seed=0)
- **Tasks:** GLUE benchmark (8 tasks) + Diagnostic stress test (4 ratios)
- **Comparison:** Side-by-side tables with original code results

**Hardware:** Requires GPU. Go to `Runtime → Change runtime type → T4 GPU`.

In [1]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not available. Go to Runtime -> Change runtime type -> T4 GPU"
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
device = "cuda"

GPU: NVIDIA A100-SXM4-80GB (85.1 GB)


In [2]:
!pip install -q torch-geometric transformers datasets scikit-learn scipy pyyaml tqdm
print("All packages installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 82.8 MB/s eta 0:00:00
All packages installed


In [3]:
import os, sys

# On Colab: clone repo; locally: already in repo root
if not os.path.exists("train.py"):
    if not os.path.exists("glot"):
        !git clone https://github.com/yosefyehoshua/glot.git
    else:
        !cd glot && git pull
    os.chdir("glot")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# Mount Google Drive for checkpointing (survives kernel crashes)
DRIVE_RESULTS_DIR = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_RESULTS_DIR = "/content/drive/MyDrive/glot_validation"
    os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
    print(f"Google Drive mounted. Checkpoints: {DRIVE_RESULTS_DIR}")
except (ImportError, Exception):
    print("Google Drive not available — results saved locally only")

import gc
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

from glot.backbone import BACKBONE_REGISTRY, load_backbone, get_backbone_config
from glot.model import create_pooler_and_head
from glot.utils import compute_metrics, load_config, GLUE_TASKS
from data.cache import make_cached_dataset, save_cache, CachedDataset
from data.glue_loader import load_glue_task, get_task_config
from data.diagnostic import generate_diagnostic_dataset
from train import train_epoch

print(f"Working directory: {os.getcwd()}")
print("All imports successful")

Mounted at /content/drive
Google Drive mounted. Checkpoints: /content/drive/MyDrive/glot_validation
Working directory: /content/glot
All imports successful


## Configuration

Hyperparameters match the original code **exactly** to ensure a fair comparison.

| Parameter | Value | Source |
|-----------|-------|--------|
| `hidden_dim` | 256 | Original `--gat_hidden_dim 256` |
| `num_layers` | 4 | Original `--num_layers 4` |
| `threshold` | 0.3 | Original `--tau 0.3` |
| `jk_mode` | cat | Original `--jk_mode cat` |
| `gnn_type` | GAT | Original uses GATConv only |
| `epochs` | 3 | Original `--epochs 3` |
| `lr` | 2e-4 | Original `--lr 2e-4` |
| `weight_decay` | 1e-5 | Original `--weight_decay 1e-5` |
| `seed` | 0 | Original `--seed 0` |
| `batch_size` | 32 | Original `--batch_size 32` |
| `max_length` | 128 | Original `--max_length 128` |

In [4]:
# === Configuration — matching original code hyperparameters ===

BACKBONES = ["bert-base-uncased", "roberta-base"]
GLUE_TASK_NAMES = ["sst2", "cola", "mrpc", "stsb", "mnli", "qnli", "rte", "wnli"]
POOLER_NAMES = ["cls", "mean", "max", "adapool", "glot"]

# Hyperparameters (matching original code exactly)
EPOCHS = 3
LR = 2e-4
WEIGHT_DECAY = 1e-5
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
SEED = 0
MAX_LENGTH = 128

# GLOT config (matching original: gat_hidden_dim=256, num_layers=4, tau=0.3)
GLOT_CONFIG = {
    "hidden_dim": 256,
    "num_gnn_layers": 4,
    "jk_mode": "cat",
    "threshold": 0.3,
    "gnn_type": "GAT",
}

# Diagnostic stress test
DIAGNOSTIC_RATIOS = [0.2, 0.5, 0.8, 0.9]

print(f"Backbones: {BACKBONES}")
print(f"GLUE tasks: {GLUE_TASK_NAMES}")
print(f"Poolers: {POOLER_NAMES}")
print(f"GLOT config: {GLOT_CONFIG}")
print(f"Training: epochs={EPOCHS}, lr={LR}, wd={WEIGHT_DECAY}, seed={SEED}")

Backbones: ['bert-base-uncased', 'roberta-base']
GLUE tasks: ['sst2', 'cola', 'mrpc', 'stsb', 'mnli', 'qnli', 'rte', 'wnli']
Poolers: ['cls', 'mean', 'max', 'adapool', 'glot']
GLOT config: {'hidden_dim': 256, 'num_gnn_layers': 4, 'jk_mode': 'cat', 'threshold': 0.3, 'gnn_type': 'GAT'}
Training: epochs=3, lr=0.0002, wd=1e-05, seed=0


In [5]:
# === Reference results from original code (reproduce_org_glot.ipynb) ===
# Two modes:
#   1. Load from Google Drive JSONs saved by reproduce_org_glot.ipynb (preferred on Colab)
#   2. Fall back to hardcoded values extracted from notebook outputs
#
# The reproduce notebook saves to /content/drive/MyDrive/glot_reproduction/:
#   - glue_results.json: {"backbone|pooler|task": {"acc": v, "mcc": v, "f1": v, "spearman": v, ...}}
#   - diagnostic_results.json: {"backbone|pooler|ratio": {"acc": v, ...}}

import glob as _glob

# --- Google Drive path (matches reproduce_org_glot.ipynb) ---
ORG_RESULTS_DIR = "/content/drive/MyDrive/glot_reproduction"

# Task -> metric key used in original code output
TASK_ORIG_KEY = {
    "sst2": "acc", "cola": "mcc", "mrpc": "f1", "stsb": "spearman",
    "mnli": "acc", "qnli": "acc", "rte": "acc", "wnli": "acc",
}
# Task -> our metric name (from compute_metrics)
TASK_METRIC_KEY = {
    "sst2": "accuracy", "cola": "mcc", "mrpc": "f1", "stsb": "spearman",
    "mnli": "accuracy", "qnli": "accuracy", "rte": "accuracy", "wnli": "accuracy",
}


def _load_org_json(path):
    """Load JSON, returning empty dict if missing or corrupt."""
    if not os.path.exists(path):
        return {}
    try:
        with open(path) as f:
            data = json.load(f)
        return data if isinstance(data, dict) else {}
    except (json.JSONDecodeError, ValueError):
        return {}


def _merge_org_results(prefix):
    """Merge results from all instance files (matches reproduce_org_glot multi-instance logic)."""
    merged = {}
    for path in sorted(_glob.glob(os.path.join(ORG_RESULTS_DIR, f"{prefix}*.json"))):
        if path.endswith(".tmp"):
            continue
        merged.update(_load_org_json(path))
    return merged


def _parse_glue_from_drive(raw):
    """Convert flat 'backbone|pooler|task' -> metrics dict to nested ORIGINAL_GLUE format."""
    result = {}
    for key, metrics in raw.items():
        parts = key.split("|")
        if len(parts) != 3:
            continue
        backbone, pooler, task = parts
        if task not in TASK_ORIG_KEY:
            continue
        metric_key = TASK_ORIG_KEY[task]
        if metric_key not in metrics:
            continue
        result.setdefault(backbone, {}).setdefault(task, {})[pooler] = {metric_key: metrics[metric_key]}
    return result


def _parse_diag_from_drive(raw):
    """Convert flat 'backbone|pooler|ratio' -> metrics dict to nested ORIGINAL_DIAG format."""
    result = {}
    for key, metrics in raw.items():
        parts = key.split("|")
        if len(parts) != 3:
            continue
        backbone, pooler, ratio_str = parts
        if "acc" not in metrics:
            continue
        try:
            ratio = float(ratio_str)
        except ValueError:
            continue
        result.setdefault(backbone, {}).setdefault(ratio, {})[pooler] = metrics["acc"]
    return result


# --- Try loading from Google Drive first ---
_drive_glue_raw = _merge_org_results("glue_results") if os.path.isdir(ORG_RESULTS_DIR) else {}
_drive_diag_raw = _merge_org_results("diagnostic_results") if os.path.isdir(ORG_RESULTS_DIR) else {}

if _drive_glue_raw:
    ORIGINAL_GLUE = _parse_glue_from_drive(_drive_glue_raw)
    print(f"Loaded GLUE reference from Drive: {ORG_RESULTS_DIR} ({len(_drive_glue_raw)} entries)")
else:
    # Hardcoded fallback: exact values from reproduce_org_glot.ipynb cached outputs
    ORIGINAL_GLUE = {
        "bert-base-uncased": {
            "sst2":  {"mean": {"acc": 0.8509}, "max": {"acc": 0.8314}, "cls": {"acc": 0.8555}, "adapool": {"acc": 0.8945}, "glot": {"acc": 0.8922}},
            "cola":  {"mean": {"mcc": 0.3501}, "max": {"mcc": 0.2826}, "cls": {"mcc": 0.3793}, "adapool": {"mcc": 0.4556}, "glot": {"mcc": 0.4783}},
            "mrpc":  {"mean": {"f1": 0.8235}, "max": {"f1": 0.8251}, "cls": {"f1": 0.8235}, "adapool": {"f1": 0.821}, "glot": {"f1": 0.8172}},
            "stsb":  {"mean": {"spearman": 0.7933}, "max": {"spearman": 0.778}, "cls": {"spearman": 0.7143}, "adapool": {"spearman": 0.833}, "glot": {"spearman": 0.8211}},
            "mnli":  {"mean": {"acc": 0.508}, "max": {"acc": 0.4614}, "cls": {"acc": 0.5078}, "adapool": {"acc": 0.5352}, "glot": {"acc": 0.5342}},
            "qnli":  {"mean": {"acc": 0.6127}, "max": {"acc": 0.5523}, "cls": {"acc": 0.5834}, "adapool": {"acc": 0.6066}, "glot": {"acc": 0.6139}},
            "rte":   {"mean": {"acc": 0.574}, "max": {"acc": 0.5054}, "cls": {"acc": 0.5668}, "adapool": {"acc": 0.5632}, "glot": {"acc": 0.5415}},
            "wnli":  {"mean": {"acc": 0.493}, "max": {"acc": 0.5493}, "cls": {"acc": 0.4507}, "adapool": {"acc": 0.3944}, "glot": {"acc": 0.5775}},
        },
        "roberta-base": {
            "sst2":  {"mean": {"acc": 0.8532}, "max": {"acc": 0.82}, "cls": {"acc": 0.8073}, "adapool": {"acc": 0.9209}, "glot": {"acc": 0.9266}},
            "cola":  {"mean": {"mcc": 0.3256}, "max": {"mcc": 0.2375}, "cls": {"mcc": 0.0}, "adapool": {"mcc": 0.4713}, "glot": {"mcc": 0.5474}},
            "mrpc":  {"mean": {"f1": 0.8251}, "max": {"f1": 0.8164}, "cls": {"f1": 0.8122}, "adapool": {"f1": 0.8239}, "glot": {"f1": 0.8246}},
            "stsb":  {"mean": {"spearman": 0.8044}, "max": {"spearman": 0.7535}, "cls": {"spearman": 0.7808}, "adapool": {"spearman": 0.8461}, "glot": {"spearman": 0.8446}},
            "mnli":  {"mean": {"acc": 0.4884}, "max": {"acc": 0.4637}, "cls": {"acc": 0.4381}, "adapool": {"acc": 0.5257}, "glot": {"acc": 0.565}},
            "qnli":  {"mean": {"acc": 0.611}, "max": {"acc": 0.5894}, "cls": {"acc": 0.5649}, "adapool": {"acc": 0.6107}, "glot": {"acc": 0.6134}},
            "rte":   {"mean": {"acc": 0.5704}, "max": {"acc": 0.5271}, "cls": {"acc": 0.556}, "adapool": {"acc": 0.5596}, "glot": {"acc": 0.5668}},
            "wnli":  {"mean": {"acc": 0.5775}, "max": {"acc": 0.5211}, "cls": {"acc": 0.5634}, "adapool": {"acc": 0.3662}, "glot": {"acc": 0.5493}},
        },
    }
    print("Using hardcoded GLUE reference (Drive not available)")

if _drive_diag_raw:
    ORIGINAL_DIAG = _parse_diag_from_drive(_drive_diag_raw)
    print(f"Loaded diagnostic reference from Drive: {ORG_RESULTS_DIR} ({len(_drive_diag_raw)} entries)")
else:
    # Hardcoded fallback: exact values from reproduce_org_glot.ipynb cached outputs
    ORIGINAL_DIAG = {
        "bert-base-uncased": {
            0.2: {"mean": 0.680, "max": 0.574, "cls": 0.708, "adapool": 0.914, "glot": 0.982},
            0.5: {"mean": 0.586, "max": 0.508, "cls": 0.582, "adapool": 0.788, "glot": 0.922},
            0.8: {"mean": 0.642, "max": 0.516, "cls": 0.572, "adapool": 0.696, "glot": 0.932},
            0.9: {"mean": 0.534, "max": 0.504, "cls": 0.676, "adapool": 0.528, "glot": 0.898},
        },
        "roberta-base": {
            0.2: {"mean": 0.736, "max": 0.616, "cls": 0.836, "adapool": 0.830, "glot": 0.996},
            0.5: {"mean": 0.646, "max": 0.600, "cls": 0.634, "adapool": 0.672, "glot": 0.890},
            0.8: {"mean": 0.678, "max": 0.590, "cls": 0.534, "adapool": 0.598, "glot": 0.748},
            0.9: {"mean": 0.572, "max": 0.512, "cls": 0.534, "adapool": 0.598, "glot": 0.662},
        },
    }
    print("Using hardcoded diagnostic reference (Drive not available)")

print(f"\nReference: {sum(len(t) for t in ORIGINAL_GLUE.values())} GLUE task×backbone combos")
print(f"Reference: {sum(len(r) for r in ORIGINAL_DIAG.values())} diagnostic ratio×backbone combos")

Loaded GLUE reference from Drive: /content/drive/MyDrive/glot_reproduction (85 entries)
Loaded diagnostic reference from Drive: /content/drive/MyDrive/glot_reproduction (40 entries)

Reference: 16 GLUE task×backbone combos
Reference: 8 diagnostic ratio×backbone combos


## Helper Functions

In [6]:
def select_pooler_type(pooler_name, backbone_type):
    """Map pooler name to the correct type for the backbone."""
    if pooler_name == "cls" and backbone_type == "decoder":
        return "eos"
    return pooler_name


@torch.no_grad()
def evaluate_epoch_flex(pooler, head, loader, task_type, device="cpu", is_regression=False):
    """Evaluate and return (predictions, labels) lists."""
    pooler.eval()
    head.eval()
    all_preds, all_labels = [], []

    for batch in loader:
        if task_type == "pair_classification":
            hs_a, mask_a, hs_b, mask_b, labels = [b.to(device) for b in batch]
            z_a = pooler(hs_a, mask_a)
            z_b = pooler(hs_b, mask_b)
            logits = head(torch.cat([z_a, z_b], dim=-1))
        else:
            hs, mask, labels = [b.to(device) for b in batch]
            z = pooler(hs, mask)
            logits = head(z)

        if is_regression:
            preds = logits.squeeze(-1)
        else:
            preds = logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    return all_preds, all_labels


def encode_split(model, split_ds, is_pair, device, batch_size=64):
    """Encode a dataset split through frozen backbone, returning an in-memory CachedDataset."""
    if is_pair:
        cols = ["input_ids_a", "attention_mask_a", "input_ids_b", "attention_mask_b", "label"]
    else:
        cols = ["input_ids", "attention_mask", "label"]
    split_ds = split_ds.select_columns(cols)
    split_ds.set_format("torch")
    loader = DataLoader(split_ds, batch_size=batch_size)

    all_hs, all_masks, all_labels = [], [], []
    all_hs_b, all_masks_b = [], []

    model.eval()
    with torch.no_grad():
        for batch in tqdm(loader, desc="  Encoding", leave=False):
            if is_pair:
                ids_a = batch["input_ids_a"].to(device)
                mask_a = batch["attention_mask_a"].to(device)
                ids_b = batch["input_ids_b"].to(device)
                mask_b = batch["attention_mask_b"].to(device)
                out_a = model(input_ids=ids_a, attention_mask=mask_a)
                out_b = model(input_ids=ids_b, attention_mask=mask_b)
                all_hs.append(out_a.last_hidden_state.cpu().float())
                all_masks.append(mask_a.cpu())
                all_hs_b.append(out_b.last_hidden_state.cpu().float())
                all_masks_b.append(mask_b.cpu())
            else:
                ids = batch["input_ids"].to(device)
                mask = batch["attention_mask"].to(device)
                out = model(input_ids=ids, attention_mask=mask)
                all_hs.append(out.last_hidden_state.cpu().float())
                all_masks.append(mask.cpu())
            all_labels.append(batch["label"])

    hs = torch.cat(all_hs)
    masks = torch.cat(all_masks)
    labels = torch.cat(all_labels)

    if is_pair:
        return CachedDataset(hs, masks, labels, torch.cat(all_hs_b), torch.cat(all_masks_b))
    return CachedDataset(hs, masks, labels)


def save_checkpoint(data, name):
    """Save intermediate results to Google Drive (if mounted) and locally."""
    local_path = f"{name}.json"
    with open(local_path, "w") as f:
        json.dump(data, f, indent=2)
    if DRIVE_RESULTS_DIR:
        drive_path = os.path.join(DRIVE_RESULTS_DIR, f"{name}.json")
        with open(drive_path, "w") as f:
            json.dump(data, f, indent=2)
        print(f"  Checkpoint saved: {drive_path}")


def load_checkpoint(name):
    """Load checkpoint from Google Drive (preferred) or local file."""
    if DRIVE_RESULTS_DIR:
        path = os.path.join(DRIVE_RESULTS_DIR, f"{name}.json")
        if os.path.exists(path):
            with open(path) as f:
                data = json.load(f)
            print(f"Resumed from Drive checkpoint: {path} ({len(data)} entries)")
            return data
    local_path = f"{name}.json"
    if os.path.exists(local_path):
        with open(local_path) as f:
            data = json.load(f)
        print(f"Resumed from local checkpoint: {local_path} ({len(data)} entries)")
        return data
    return None


print("Helpers defined")

Helpers defined


## Experiment 1: GLUE Benchmark

For each encoder backbone: load model, encode each task, train all 5 poolers with **original hyperparameters**, then compare against original code results.

Note: The original code uses no subsampling for encoder backbones (768-dim fits in memory). We follow the same approach.

In [ ]:
torch.manual_seed(SEED)
glue_results = {}  # (backbone, task, pooler) -> score
completed_backbones = []

# Resume from checkpoint if available
_ckpt = load_checkpoint("validation_glue_results")
if _ckpt:
    for str_key, score in _ckpt.items():
        b, t, p = str_key.split("|")
        glue_results[(b, t, p)] = score
    for backbone_name in BACKBONES:
        if all((backbone_name, t, p) in glue_results for t in GLUE_TASK_NAMES for p in POOLER_NAMES):
            completed_backbones.append(backbone_name)
    print(f"Resuming: {len(glue_results)} experiments, {len(completed_backbones)} backbones complete")

for backbone_name in BACKBONES:
    if backbone_name in completed_backbones:
        print(f"\nSkipping {backbone_name} (already complete from checkpoint)")
        continue

    bcfg = get_backbone_config(backbone_name)
    input_dim = bcfg["hidden_dim"]

    print(f"\n{'#'*60}")
    print(f"# {backbone_name} (dim={input_dim})")
    print(f"{'#'*60}")

    model, tokenizer, _ = load_backbone(backbone_name, device=device)

    for task_name in GLUE_TASK_NAMES:
        task_cfg = GLUE_TASKS[task_name]
        is_pair = task_cfg["type"] == "pair"
        task_type = "pair_classification" if is_pair else "classification"
        is_regression = task_cfg["num_classes"] == 1
        metric_name = task_cfg["metric"]

        # Skip if all poolers done for this (backbone, task)
        if all((backbone_name, task_name, p) in glue_results for p in POOLER_NAMES):
            print(f"\n  === {task_name} — all poolers done (checkpoint) ===")
            continue

        print(f"\n  === {task_name} ({metric_name}) ===")

        dataset = load_glue_task(task_name, tokenizer, MAX_LENGTH)
        train_split = dataset["train"]

        # Subsample large datasets (matching original: QQP, MNLI, QNLI at 20K)
        max_samples = 20000
        if len(train_split) > max_samples:
            train_split = train_split.shuffle(seed=SEED).select(range(max_samples))
            print(f"  Subsampled train to {max_samples}")

        # Determine validation split key (MNLI uses validation_matched)
        val_key = "validation_matched" if task_name == "mnli" else "validation"
        val_split = dataset[val_key]

        print(f"  Encoding train ({len(train_split)})...")
        train_ds = encode_split(model, train_split, is_pair, device)
        print(f"  Encoding val ({len(val_split)})...")
        val_ds = encode_split(model, val_split, is_pair, device)

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=EVAL_BATCH_SIZE)

        for pooler_name in POOLER_NAMES:
            if (backbone_name, task_name, pooler_name) in glue_results:
                print(f"    {pooler_name}: {glue_results[(backbone_name, task_name, pooler_name)]:.2f} (checkpoint)")
                continue

            actual_pooler = select_pooler_type(pooler_name, bcfg["type"])
            glot_cfg = GLOT_CONFIG if actual_pooler == "glot" else None
            pooler, head = create_pooler_and_head(
                pooler_type=actual_pooler,
                input_dim=input_dim,
                num_classes=task_cfg["num_classes"],
                task_type=task_type,
                glot_config=glot_cfg,
            )
            pooler.to(device)
            head.to(device)

            params = list(pooler.parameters()) + list(head.parameters())
            optimizer = Adam(params, lr=LR, weight_decay=WEIGHT_DECAY)

            if is_regression:
                loss_fn = lambda logits, labels: nn.MSELoss()(logits.squeeze(-1), labels.float())
            else:
                loss_fn = nn.CrossEntropyLoss()

            torch.manual_seed(SEED)
            best_score = -float("inf")
            for epoch in range(EPOCHS):
                avg_loss = train_epoch(pooler, head, train_loader, optimizer, loss_fn, task_type, device)
                preds, labels = evaluate_epoch_flex(pooler, head, val_loader, task_type, device, is_regression)
                score = compute_metrics(preds, labels, metric_name)
                best_score = max(best_score, score)
                print(f"    {pooler_name} ep{epoch+1}: loss={avg_loss:.4f} {metric_name}={score:.2f}")

            glue_results[(backbone_name, task_name, pooler_name)] = best_score

            del pooler, head, optimizer
            torch.cuda.empty_cache()

        del train_ds, val_ds, train_loader, val_loader
        gc.collect()

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    completed_backbones.append(backbone_name)

    # Checkpoint after each backbone
    _serializable = {f"{b}|{t}|{p}": s for (b, t, p), s in glue_results.items()}
    save_checkpoint(_serializable, "validation_glue_results")

print(f"\n\nGLUE complete: {len(glue_results)} experiments")

### GLUE Results: Side-by-Side Comparison

Our scores are in ×100 format (from `compute_metrics`). Original scores are in 0-1 format.
Delta = Ours − Original×100. Positive = we scored higher.

In [ ]:
# Per-backbone comparison tables
for backbone_name in BACKBONES:
    print(f"\n{'='*80}")
    print(f"  {backbone_name}")
    print(f"{'='*80}")

    rows = []
    for pooler_name in POOLER_NAMES:
        row = {"Pooler": pooler_name.upper()}
        for task_name in GLUE_TASK_NAMES:
            metric = TASK_METRIC_KEY[task_name]
            orig_key = TASK_ORIG_KEY[task_name]

            ours = glue_results.get((backbone_name, task_name, pooler_name), float("nan"))

            orig_raw = ORIGINAL_GLUE.get(backbone_name, {}).get(task_name, {}).get(pooler_name, {}).get(orig_key, None)
            if orig_raw is not None:
                orig_scaled = orig_raw * 100
                delta = ours - orig_scaled
                row[task_name] = f"{ours:.1f} ({delta:+.1f})"
            else:
                row[task_name] = f"{ours:.1f}"
        rows.append(row)

    df = pd.DataFrame(rows).set_index("Pooler")
    print(df.to_string())
    print()

In [ ]:
# Aggregate delta analysis: how close are we to the original code?
print("GLUE Delta Analysis: Ours vs Original (×100 scale)")
print("="*60)

deltas = []
for backbone_name in BACKBONES:
    for task_name in GLUE_TASK_NAMES:
        orig_key = TASK_ORIG_KEY[task_name]
        for pooler_name in POOLER_NAMES:
            ours = glue_results.get((backbone_name, task_name, pooler_name))
            orig_raw = ORIGINAL_GLUE.get(backbone_name, {}).get(task_name, {}).get(pooler_name, {}).get(orig_key)
            if ours is not None and orig_raw is not None:
                delta = ours - orig_raw * 100
                deltas.append({
                    "backbone": backbone_name.split("/")[-1],
                    "task": task_name,
                    "pooler": pooler_name,
                    "ours": ours,
                    "original": orig_raw * 100,
                    "delta": delta,
                    "abs_delta": abs(delta),
                })

df_deltas = pd.DataFrame(deltas)
print(f"\nTotal comparisons: {len(df_deltas)}")
print(f"Mean absolute delta: {df_deltas['abs_delta'].mean():.2f}")
print(f"Max absolute delta: {df_deltas['abs_delta'].max():.2f}")
print(f"Within ±2%: {(df_deltas['abs_delta'] <= 2.0).sum()}/{len(df_deltas)}")
print(f"Within ±5%: {(df_deltas['abs_delta'] <= 5.0).sum()}/{len(df_deltas)}")

# Per-pooler summary
print(f"\nPer-pooler mean |delta|:")
for pooler_name in POOLER_NAMES:
    subset = df_deltas[df_deltas["pooler"] == pooler_name]
    print(f"  {pooler_name:>8}: {subset['abs_delta'].mean():.2f}")

# Largest discrepancies
print(f"\nLargest discrepancies (|delta| > 5):")
big = df_deltas[df_deltas["abs_delta"] > 5.0].sort_values("abs_delta", ascending=False)
if len(big) > 0:
    print(big[["backbone", "task", "pooler", "ours", "original", "delta"]].to_string(index=False))
else:
    print("  None — all within ±5%!")

In [ ]:
# Scatter plot: our score vs original score (perfect match = diagonal)
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

colors = {"cls": "#1f77b4", "mean": "#ff7f0e", "max": "#2ca02c", "adapool": "#d62728", "glot": "#9467bd"}
markers = {"bert-base-uncased": "o", "roberta-base": "s"}

for _, row in df_deltas.iterrows():
    ax.scatter(
        row["original"], row["ours"],
        c=colors[row["pooler"]], marker=markers.get(row["backbone"], "o"),
        s=60, alpha=0.7,
    )

# Diagonal line (perfect match)
lim = [min(df_deltas["original"].min(), df_deltas["ours"].min()) - 5,
       max(df_deltas["original"].max(), df_deltas["ours"].max()) + 5]
ax.plot(lim, lim, "k--", alpha=0.3, label="Perfect match")
ax.fill_between(lim, [l - 5 for l in lim], [l + 5 for l in lim], alpha=0.1, color="gray", label="±5% band")

# Legend for poolers
for p, c in colors.items():
    ax.scatter([], [], c=c, label=p.upper(), s=60)
# Legend for backbones
for b, m in markers.items():
    ax.scatter([], [], c="gray", marker=m, label=b.split("/")[-1], s=60)

ax.set_xlabel("Original Code Score (×100)", fontsize=12)
ax.set_ylabel("Our Structured Code Score (×100)", fontsize=12)
ax.set_title("GLUE Validation: Structured vs Original", fontsize=14, fontweight="bold")
ax.legend(fontsize=9, loc="upper left")
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("validation_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

## Experiment 2: Diagnostic Stress Test

Signal dilution experiment with original hyperparameters. Comparing our structured code against original code at each distractor ratio.

In [ ]:
# Diagnostic experiment: encode synthetic data through backbone, train poolers, compare accuracy
# Uses same hyperparameters as original code (hidden=256, layers=4, tau=0.3, seed=0)

DIAG_TRAIN_SAMPLES = 10000
DIAG_TEST_SAMPLES = 2000
DIAG_SEQ_LENGTH = 256
DIAG_MAX_TOKEN_LENGTH = 512
DIAG_EPOCHS = 3  # matching original code
DIAG_LR = 2e-4
DIAG_BATCH_SIZE = 32

diagnostic_results = {}  # (backbone, ratio, pooler) -> accuracy (×100)

# Resume from checkpoint if available
_diag_ckpt = load_checkpoint("validation_diag_results")
if _diag_ckpt:
    for str_key, score in _diag_ckpt.items():
        b, r, p = str_key.split("|")
        diagnostic_results[(b, float(r), p)] = score
    print(f"Resuming: {len(diagnostic_results)} diagnostic experiments done")

for backbone_name in BACKBONES:
    bcfg = get_backbone_config(backbone_name)
    input_dim = bcfg["hidden_dim"]

    # Check if all experiments for this backbone are done
    all_done = all(
        (backbone_name, r, p) in diagnostic_results
        for r in DIAGNOSTIC_RATIOS for p in POOLER_NAMES
    )
    if all_done:
        print(f"\nSkipping {backbone_name} (all done from checkpoint)")
        continue

    print(f"\n{'#'*60}")
    print(f"# Diagnostic: {backbone_name} (dim={input_dim})")
    print(f"{'#'*60}")

    model, tokenizer, _ = load_backbone(backbone_name, device=device)

    for ratio in DIAGNOSTIC_RATIOS:
        # Check if all poolers for this (backbone, ratio) are done
        ratio_done = all(
            (backbone_name, ratio, p) in diagnostic_results for p in POOLER_NAMES
        )
        if ratio_done:
            print(f"\n  --- {int(ratio*100)}% distractors — all done (checkpoint) ---")
            continue

        print(f"\n  --- {int(ratio*100)}% distractors ---")

        # Generate synthetic data
        train_data = generate_diagnostic_dataset(
            num_samples=DIAG_TRAIN_SAMPLES, seq_length=DIAG_SEQ_LENGTH,
            distractor_ratio=ratio, seed=SEED,
        )
        test_data = generate_diagnostic_dataset(
            num_samples=DIAG_TEST_SAMPLES, seq_length=DIAG_SEQ_LENGTH,
            distractor_ratio=ratio, seed=SEED + 1,
        )

        train_texts = [t for t, _ in train_data]
        train_labels = torch.tensor([l for _, l in train_data])
        test_texts = [t for t, _ in test_data]
        test_labels = torch.tensor([l for _, l in test_data])

        # Encode through frozen backbone
        print("    Encoding train...")
        train_hs_list, train_mask_list = [], []
        model.eval()
        with torch.no_grad():
            for i in tqdm(range(0, len(train_texts), 16), desc="    train", leave=False):
                batch_texts = train_texts[i:i+16]
                enc = tokenizer(
                    batch_texts, truncation=True, max_length=DIAG_MAX_TOKEN_LENGTH,
                    padding="max_length", return_tensors="pt",
                ).to(device)
                out = model(**enc)
                train_hs_list.append(out.last_hidden_state.cpu().float())
                train_mask_list.append(enc["attention_mask"].cpu())
        train_hs = torch.cat(train_hs_list)
        train_masks = torch.cat(train_mask_list)
        del train_hs_list, train_mask_list

        print("    Encoding test...")
        test_hs_list, test_mask_list = [], []
        with torch.no_grad():
            for i in tqdm(range(0, len(test_texts), 16), desc="    test", leave=False):
                batch_texts = test_texts[i:i+16]
                enc = tokenizer(
                    batch_texts, truncation=True, max_length=DIAG_MAX_TOKEN_LENGTH,
                    padding="max_length", return_tensors="pt",
                ).to(device)
                out = model(**enc)
                test_hs_list.append(out.last_hidden_state.cpu().float())
                test_mask_list.append(enc["attention_mask"].cpu())
        test_hs = torch.cat(test_hs_list)
        test_masks = torch.cat(test_mask_list)
        del test_hs_list, test_mask_list
        gc.collect()

        # Train each pooler
        for pooler_name in POOLER_NAMES:
            if (backbone_name, ratio, pooler_name) in diagnostic_results:
                print(f"    {pooler_name}: {diagnostic_results[(backbone_name, ratio, pooler_name)]:.1f}% (checkpoint)")
                continue

            actual_pooler = select_pooler_type(pooler_name, bcfg["type"])
            glot_cfg = GLOT_CONFIG if actual_pooler == "glot" else None
            pooler, head = create_pooler_and_head(
                pooler_type=actual_pooler,
                input_dim=input_dim,
                num_classes=2,
                task_type="classification",
                glot_config=glot_cfg,
            )
            pooler.to(device)
            head.to(device)

            train_ds = TensorDataset(train_hs, train_masks, train_labels)
            test_ds = TensorDataset(test_hs, test_masks, test_labels)
            train_loader = DataLoader(train_ds, batch_size=DIAG_BATCH_SIZE, shuffle=True)
            test_loader = DataLoader(test_ds, batch_size=64)

            params = list(pooler.parameters()) + list(head.parameters())
            optimizer = Adam(params, lr=DIAG_LR, weight_decay=WEIGHT_DECAY)
            loss_fn = nn.CrossEntropyLoss()
            torch.manual_seed(SEED)

            for epoch in range(DIAG_EPOCHS):
                pooler.train()
                head.train()
                for hs, masks, labels in train_loader:
                    hs, masks, labels = hs.to(device), masks.to(device), labels.to(device)
                    optimizer.zero_grad()
                    z = pooler(hs, masks)
                    logits = head(z)
                    loss = loss_fn(logits, labels)
                    loss.backward()
                    optimizer.step()

            # Evaluate
            pooler.eval()
            head.eval()
            all_preds, all_labels_list = [], []
            with torch.no_grad():
                for hs, masks, labels in test_loader:
                    hs, masks, labels = hs.to(device), masks.to(device), labels.to(device)
                    z = pooler(hs, masks)
                    logits = head(z)
                    all_preds.extend(logits.argmax(dim=-1).cpu().tolist())
                    all_labels_list.extend(labels.cpu().tolist())

            accuracy = compute_metrics(all_preds, all_labels_list, "accuracy")
            diagnostic_results[(backbone_name, ratio, pooler_name)] = accuracy
            print(f"    {pooler_name}: {accuracy:.1f}%")

            del pooler, head, optimizer
            torch.cuda.empty_cache()

        del train_hs, train_masks, test_hs, test_masks
        gc.collect()

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    # Checkpoint after each backbone
    _serializable = {f"{b}|{r}|{p}": s for (b, r, p), s in diagnostic_results.items()}
    save_checkpoint(_serializable, "validation_diag_results")

print(f"\n\nDiagnostic complete: {len(diagnostic_results)} experiments")

### Diagnostic Results: Side-by-Side Comparison

Our scores are in ×100 format. Original scores are in 0-1 format.
Delta = Ours − Original×100. Positive = we scored higher.

In [ ]:
# Per-backbone, per-ratio comparison tables
for backbone_name in BACKBONES:
    print(f"\n{'='*80}")
    print(f"  {backbone_name}")
    print(f"{'='*80}")

    for ratio in DIAGNOSTIC_RATIOS:
        print(f"\n  {int(ratio*100)}% Distractors:")
        header = f"  {'Pooler':<10} {'Ours':>8} {'Original':>10} {'Delta':>8}"
        print(header)
        print(f"  {'-'*38}")

        for pooler_name in POOLER_NAMES:
            ours = diagnostic_results.get((backbone_name, ratio, pooler_name), float("nan"))
            orig_raw = ORIGINAL_DIAG.get(backbone_name, {}).get(ratio, {}).get(pooler_name, None)

            if orig_raw is not None:
                orig_scaled = orig_raw * 100
                delta = ours - orig_scaled
                print(f"  {pooler_name:<10} {ours:>7.1f}% {orig_scaled:>9.1f}% {delta:>+7.1f}")
            else:
                print(f"  {pooler_name:<10} {ours:>7.1f}%       --      --")

# Aggregate diagnostic delta analysis
print(f"\n\n{'='*80}")
print("  Diagnostic Delta Analysis: Ours vs Original (×100 scale)")
print(f"{'='*80}")

diag_deltas = []
for backbone_name in BACKBONES:
    for ratio in DIAGNOSTIC_RATIOS:
        for pooler_name in POOLER_NAMES:
            ours = diagnostic_results.get((backbone_name, ratio, pooler_name))
            orig_raw = ORIGINAL_DIAG.get(backbone_name, {}).get(ratio, {}).get(pooler_name)
            if ours is not None and orig_raw is not None:
                delta = ours - orig_raw * 100
                diag_deltas.append({
                    "backbone": backbone_name.split("/")[-1],
                    "ratio": ratio,
                    "pooler": pooler_name,
                    "ours": ours,
                    "original": orig_raw * 100,
                    "delta": delta,
                    "abs_delta": abs(delta),
                })

df_diag_deltas = pd.DataFrame(diag_deltas)
print(f"\nTotal comparisons: {len(df_diag_deltas)}")
print(f"Mean absolute delta: {df_diag_deltas['abs_delta'].mean():.2f}")
print(f"Max absolute delta: {df_diag_deltas['abs_delta'].max():.2f}")
print(f"Within ±2%: {(df_diag_deltas['abs_delta'] <= 2.0).sum()}/{len(df_diag_deltas)}")
print(f"Within ±5%: {(df_diag_deltas['abs_delta'] <= 5.0).sum()}/{len(df_diag_deltas)}")

# Per-pooler breakdown
print(f"\nPer-pooler mean |delta|:")
for pooler_name in POOLER_NAMES:
    subset = df_diag_deltas[df_diag_deltas["pooler"] == pooler_name]
    print(f"  {pooler_name:>8}: {subset['abs_delta'].mean():.2f}")

# Largest discrepancies
print(f"\nLargest discrepancies (|delta| > 5):")
big = df_diag_deltas[df_diag_deltas["abs_delta"] > 5.0].sort_values("abs_delta", ascending=False)
if len(big) > 0:
    print(big[["backbone", "ratio", "pooler", "ours", "original", "delta"]].to_string(index=False))
else:
    print("  None — all within ±5%!")

In [ ]:
# 2x2 grid: our results vs original results at each distractor ratio
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors = {"cls": "#1f77b4", "mean": "#ff7f0e", "max": "#2ca02c", "adapool": "#d62728", "glot": "#9467bd"}
bar_width = 0.15

for ax, ratio in zip(axes.flat, DIAGNOSTIC_RATIOS):
    x = np.arange(len(POOLER_NAMES))

    # Our results
    ours_vals = []
    orig_vals = []
    for pooler_name in POOLER_NAMES:
        for backbone_name in BACKBONES:
            ours = diagnostic_results.get((backbone_name, ratio, pooler_name))
            orig_raw = ORIGINAL_DIAG.get(backbone_name, {}).get(ratio, {}).get(pooler_name)
            if ours is not None and orig_raw is not None:
                break
        ours_vals.append(ours if ours is not None else 0)
        orig_vals.append(orig_raw * 100 if orig_raw is not None else 0)

    # Plot grouped bars for each backbone
    backbone_x = np.arange(len(BACKBONES))
    for bi, backbone_name in enumerate(BACKBONES):
        ours_by_pooler = []
        orig_by_pooler = []
        for pooler_name in POOLER_NAMES:
            ours = diagnostic_results.get((backbone_name, ratio, pooler_name), 0)
            orig_raw = ORIGINAL_DIAG.get(backbone_name, {}).get(ratio, {}).get(pooler_name, 0)
            ours_by_pooler.append(ours)
            orig_by_pooler.append(orig_raw * 100)

        x_pooler = np.arange(len(POOLER_NAMES))
        short_name = backbone_name.split("/")[-1][:8]
        ax.bar(x_pooler - 0.15 + bi * 0.0, ours_by_pooler, 0.25, alpha=0.0)  # invisible spacer

    # Simpler: one bar pair per pooler (averaged across backbones)
    ax.clear()
    for pi, pooler_name in enumerate(POOLER_NAMES):
        ours_list, orig_list = [], []
        for backbone_name in BACKBONES:
            ours = diagnostic_results.get((backbone_name, ratio, pooler_name))
            orig_raw = ORIGINAL_DIAG.get(backbone_name, {}).get(ratio, {}).get(pooler_name)
            if ours is not None and orig_raw is not None:
                ours_list.append(ours)
                orig_list.append(orig_raw * 100)

        ours_mean = np.mean(ours_list) if ours_list else 0
        orig_mean = np.mean(orig_list) if orig_list else 0

        ax.bar(pi - 0.15, ours_mean, 0.28, color=colors[pooler_name], alpha=0.9, label="Ours" if pi == 0 else "")
        ax.bar(pi + 0.15, orig_mean, 0.28, color=colors[pooler_name], alpha=0.4, label="Original" if pi == 0 else "")

    ax.set_title(f"{int(ratio*100)}% Distractors", fontsize=13, fontweight="bold")
    ax.set_ylabel("Accuracy (%)", fontsize=11)
    ax.set_xticks(range(len(POOLER_NAMES)))
    ax.set_xticklabels([p.upper() for p in POOLER_NAMES], fontsize=10)
    ax.set_ylim(40, 105)
    ax.grid(axis="y", alpha=0.3)

    # Add "Ours" / "Original" to legend
    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(facecolor="gray", alpha=0.9, label="Ours (structured)"),
        Patch(facecolor="gray", alpha=0.4, label="Original code"),
    ], fontsize=9, loc="upper left")

fig.suptitle("Diagnostic Stress Test: Structured vs Original (averaged across backbones)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("validation_diagnostic.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Save all results to JSON (local + Google Drive)
all_results = {
    "glue": {f"{b}|{t}|{p}": s for (b, t, p), s in glue_results.items()},
    "diagnostic": {f"{b}|{r}|{p}": s for (b, r, p), s in diagnostic_results.items()},
    "config": {
        "backbones": BACKBONES,
        "glue_tasks": GLUE_TASK_NAMES,
        "poolers": POOLER_NAMES,
        "diagnostic_ratios": DIAGNOSTIC_RATIOS,
        "epochs": EPOCHS,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "seed": SEED,
        "glot_config": GLOT_CONFIG,
    },
}

save_checkpoint(all_results, "validation_all_results")
print(f"All results saved")
print(f"  GLUE: {len(glue_results)} experiments")
print(f"  Diagnostic: {len(diagnostic_results)} experiments")

## Summary

This notebook validated that our **structured GLOT implementation** produces results consistent with the **original monolithic code** from [ipsitmantri/GLOT](https://github.com/ipsitmantri/GLOT).

### What was compared
- **GLUE Benchmark:** 8 tasks × 2 encoder backbones × 5 poolers = 80 experiments
- **Diagnostic Stress Test:** 4 distractor ratios × 2 backbones × 5 poolers = 40 experiments
- **Hyperparameters:** Matched exactly (hidden=256, layers=4, τ=0.3, epochs=3, seed=0, wd=1e-5)

### How to interpret results
- **Small deltas (±2-3%)** are expected due to non-deterministic GPU operations (cuDNN, scatter ops in PyG)
- **Consistent ranking** of poolers (GLOT > AdaPool > baselines) validates architectural correctness
- **Key check:** GLOT should still be the best or near-best pooler at high distractor ratios

### Architecture changes validated
| Component | Change | Impact |
|-----------|--------|--------|
| Graph construction | Added edge weights (`edge_attr`) | Same graph structure, weights used by GAT |
| TokenGNN | Removed input projection, raw d→GNN | JK output = d + K×p (was p×(K+1)) |
| Readout | Scaled scorer hidden dim | `max(128, dim//2)` |
| GLOTPooler | Moved to `model.py` | No functional change |
| GNN types | Added GCN/GIN/GINE | Only GAT used here (matching original) |